# GMRESについて

## 概要 (後で書く)

### Krylov部分空間

解く問題は $A x = b$ という連立一次方程式である。<br>
ここで A は $n \times n$ 次元の行列、$x, b$ はそれぞれ $n$ 次元のベクトルとする。

GMRESでは近似初期解を $x_0$ として 近似解を $x_m = x_0 + z_m$ としたときにこの$z_m$ をKrylov部分空間 $\mathcal{K}_m = \text{span}\{r_0, A r_0, A^2 r_0, \cdots A^{(m-1)} r_0\}$ の中で繰り返し計算で探すことで、元の連立一次方程式を効率よく解く。


このKrylov部分空間だけを見てみても連立方程式を解くことイメージが繋がりにくいので、この先の Arnoldi 法でつながりを見ていく。

###  Arnoldi法

#### Step 0

解の候補を$x_0$ として $r_0 = b - Ax_0$ とし、<br>
$v_1 = r_0 / ||r_0||$ として、r_0方向の単位ベクトル $v_1$ を作る

このとき $K_1= \text{span}\{v_1 \} = \text{span}\{r_0\}$<br>
よって、$v_1$ がKrylov部分空間の基底になる。

#### Step 1

$h_{11} = v_1^T A v_1$ として、$A v_1$ と $v_1$ の内積から、$A v_1$が基底 $v_1$にどれだけ乗っているかを計算する。ここで$A v_1$ はベクトル $v_1$ に線形変換 $A$ を作用させて得られる新しいベクトルである。

$w = A v_1 - h_{11} v_1$ として、$A v_1$方向から$v_1$を引き、$v_1$に直交するベクトルを作成する。

$h_{21} = ||w||$ として、$v_1$に直交するベクトルの大きさを求める。

$v_2 = w / h_{21}$ として、$v_1$に直交する単位ベクトル $v_2$ を求める。

このとき 新たな基底として$v_2$を採用し $K_2= \text{span}\{v_1 , v_2\}$<br>
よって、$v_1, v_2$がKrylov部分空間の基底になる。

Arnoldi法では、各ステップで $A v_j$ を既存の基底と新しい基底の線形結合で表す。このとき表れる係数 $h_{ij}$ を列ごとに並べた行列が上ヘッセンベルグ行列 $H$ である。

$$
\boxed{
A v_1 = h_{11} v_1 + h_{21} v_2
}
$$

この係数を一列目に並べると、

$$
\bar{H}_1 = \begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

-------------------------------------------------------------

##### Krylov部分空間 $\{r0 , A r0 \}$ との関係

部分空間 $\text{span}$ の定義は
$\text{span}\{x_1, x_2\}= \{c_1 x_1 + c_2 x_2 | c_1 , c_2 \in \R\}$であり、$\text{span}\{x_1, x_2\}$ は任意の $c1,c2$ を係数としたベクトル $c_1 x_1 + c_2 x_2$ の線形結合の集合である。

集合が$A,B$ 二つあり、$A=B$を示すには、 $A \subseteq B$ と $B \subseteq A$ を示す必要がある。

ここでは、$\text{span}\{v_1, v_2\}$ = $\text{span}\{r_0, A r_0 \}$を示す。

##### $\text{span}\{v_1, v_2\}$ $\subseteq$ $\text{span}\{r_0, A r_0 \}$ を示す

$\alpha = 1/||r_0||$とすると、$v_1 = \alpha r_0$である。

また、$v_2 = (A v_1 - h_{11} v_1)/h_{21}$ であり、$\beta = - \alpha h_{11}/h_{21}, \gamma = \alpha / h_{21}$ とすると、$v_2 = \beta r_0 + \gamma A r_0$になる。<br>
任意のベクトルは 任意の $c_1, c_2$に対して $c_1 v_1 + c_2 v_2$ と書ける。<br>
$v_1,v_2$を代入すると以下となる。

$$
c_1 \alpha r_0 + c_2 (\beta r_0 + \gamma A r_0) = (c_1 \alpha + c_2 \beta) r_0 + c_2 \gamma A r_0
$$

よって $c_1 v_1 + c_2 v_2 \in \text{span}\{ r_0, A r_0\}$ であり、

$$
\text{span}\{v_1, v_2\} \subseteq \text{span}\{r_0, A r_0 \}
$$

##### $\text{span}\{r_0, A r_0\}$ $\subseteq$ $\text{span}\{v_1, v_2 \}$ を示す

$v_2 = (A v_1 - h_{11} v_1)/h_{21}$より、$A v_1 = h_{11} v_1 + h_{21} v_2$ となる。<br>
$v_1 = \alpha r_0$ より、$A v_1 = \alpha A r_0$ である。
よって $A r_0 = (h_{11} v_1 + h_{21} v_2) /\alpha$ となり $A r_0$ は $v_1, v_2$の線形結合で表すことが出来る。
任意のベクトルは任意の $d_1, d_2$ に対して $d_1 r_0 + d_2 A r_0$ と書ける。<br>
$r_0, A r_0$ を代入すると以下となる。

$$
d_1 r_0 + d_2 A r_0 = \frac{d_1}{\alpha} v_1 + \frac{d_2 (h_{11} v_1 + h_{21} v_2)}{\alpha} =  \left(\frac{d_1}{\alpha} +\frac{d_2 h_{11}}{\alpha}\right) v_1 + \frac{d_2 h_{21}}{\alpha} v_2 
$$

よって、$d_1 r_0 + d_2 A r_0 \in \text{span} \{v_1, v_2\}$であり、

$$
\text{span}\{r_0, A r_0\} \subseteq \text{span}\{v_1, v_2 \}
$$

以上より、以下を示せる。


$$
\text{span}\{v_1, v_2\} = \text{span}\{r_0, A r_0 \}
$$

-------------------------------------------------------------

### Step 2

$h_{12} = v_1^T A v_2$ として、$A v_2$と $v_1$ の内積から、$A v_2$が基底$v_1$にどれだけ乗っているかを計算する。<br>
$h_{22} = v_2^T A v_2$ として、$A v_2$と $v_2$ の内積から、$A v_2$が基底$v_2$にどれだけ乗っているかを計算する。<br>
ここで$A v_2$ ベクトル $v_2$ に線形変換$A$を作用させて得られる新たなベクトルである。

$w = A v_2 - h_{12} v_1 - h_{22} v_2$ として、 $A v_2$から$v_1, v_2$要素を引き、$v_1, v_2$に直交するベクトルを作成する。

$h_{32} = ||w||$ として、$v_1, v_2$に直交するベクトルの大きさを求める。<br>
$v_3 = w / h_{32}$ として、$v_1, v_2$に直交する単位ベクトル $v_3$ を求める。

このとき 新たな基底として $v_3$を採用し、$K_3= \text{span}\{v_1 , v_2, v_3\}$<br>
よって、$v_1, v_2, v_3$がKrylov部分空間の基底になる。

このStepで得られる $A v_2$の展開は

$$
\boxed{
A v_2 = h_{12} v_1 + h_{22} v_2 + h_{32} v_3
}
$$

この式の係数を上ヘッセンベルグ行列の第２列に追加すると

$$
\bar{H}_2 = \begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$


ここでも Step 1 と同様に $\text{span} \{ v_1, v_2, v_3\} = \text{span} \{r_0, A r_0, A^2 r_0 \}$ と示せるが割愛する。

ここでこれまで得られた式を再掲すると以下となる。

$$
\begin{aligned}
A v_1 &= h_{11} v_1 + h_{21} v_2 \\
A v_2 &= h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{aligned}
$$

これを横に並べると、左辺は

$$
\begin{bmatrix} A v_1 & A v_2 \end{bmatrix} = A \begin{bmatrix} v_1 & v_2 \end{bmatrix} 
$$

右辺は以下となる。
$$
\begin{bmatrix}
h_{11} v_1 + h_{21} v_2 & h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{bmatrix} = 
\begin{bmatrix}v_1 & v_2 & v_3 \end{bmatrix}
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

ここで$V_2 = \begin{bmatrix} v_1 & v_2 \end{bmatrix},V_3 = \begin{bmatrix} v_1 & v_2 & v_3 \end{bmatrix}$ また$\bar{H}_2$を以下とすると、

$$
\bar{H}_2 = 
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

$$
\boxed{
A V_2 = V_3 \bar{H}_2
}
$$

#### Arnoldi法の一般化

Arnoldi法を$m$ ステップまで実行する。第$j$ステップ ($j = 1,\cdots, m$)では、次の計算を行う。

$$
\begin{aligned}
h_{ij} &= v_i^T A v_j \space , \space (i = 1,\cdots , j)\\
w &= A v_j - \sum_{i=1}^j h_{ij} v_i \\
h_{j+1,j} &= ||w|| \\
v_{j+1} &= \frac{w}{h_{j+1, j}}
\end{aligned}
$$

これを$j=1,\cdots , m$ について繰り返す。

そしてKrylov部分空間は $\text{span}\{v_1, \cdots , v_{m+1}\} = \text{span}\{r_0 , \cdots , A^m r_0\}$ となる。

また、上ヘッセンベルグ行列は以下となる。

$$
\bar{H}_m = 
\begin{bmatrix}
h_{11} & h_{12} & h_{13} & \cdots & h_{1m} \\
h_{21} & h_{22} & h_{23} & \cdots & h_{2m} \\
0 & h_{32} & h_{33} & \cdots & h_{3m} \\
0 & 0 &  h_{43} & \cdots & h_{4m} \\
\vdots & \vdots & \ddots & \ddots & \vdots \\
0 & 0 & \cdots & h_{m,m-1} & h_{mm} \\
0 & 0 & \cdots & 0 & h_{m+1, m}
\end{bmatrix}
$$

各ステップで得られた $A v_j = \sum_{i=1}^{j+1} h_{ij} v_i$ を横に並べて整理すると、以下の関係を得る。

$$
\boxed{
A V_m = V_{m+1} \bar{H}_m
}
$$

$\bar{H}_m$ は $(m+1) \times m$ の行列であることに注意する。


Arnoldi法で生成される基底 $v_1, v_2 , \cdots$ はすべて $\R^n$ のベクトルであり、互いに正規直交する。$n$次元空間内で線形独立なベクトルは最大でも$n$本であるため、Krylov部分空間の次元およびArnoldi法の反復回数の上限は $n$ 回である。

ただし途中で

$$
h_{j+1, j} = 0
$$ 

となった場合、それ以上の独立な基底を生成できなくなったことを意味し、計算は打ち切られる。

実際の大規模問題では計算量を抑えるため、通常は $m \ll n$ として打ち切る。

Arnoldi法は、残差 $r_0$ から始めて、行列 $A$ を作用させることでKrylov部分空間を少しずつ拡張し、その空間の正規直交基底 $V_m$ を生成する方法である。同時に、$A$ をこの基底上で表現した小さな行列 $\bar{H}_m$ が得られ、

$$
AV_m = V{m+1}\bar{H}_m
$$

という関係が成立する。GMRESでは、この関係を利用して、元の $n$ 次元の問題を $m$ 次元程度の小さな最小二乗問題へ変換する。

### QR分解とGives回転